In [1]:
from tqdm import tqdm
import torch
from utils.models import load_model
from utils.dataset_builder import AbstractTask
from pathlib import Path
from utils.globals import RESULTS_DIR; RESULTS_DIR: Path
import pandas as pd


def get_acc_df(model, dataset) -> pd.DataFrame:
    # Run model
    preds = []
    for x, _ in tqdm(dataset):
        with torch.no_grad():
            with model.trace(x):
                top_toks = model.next_token_probs.argmax(dim=1).save()
        preds.extend(model.tokenizer.batch_decode(top_toks))

    # Get df
    df = dataset.to_dataframe()
    df['pred'] = preds
    df['correct'] = df['completion'] == df['pred']
    return df


model_name = "Qwen/Qwen2.5-72B-Instruct" # "meta-llama/Llama-3.3-70B-Instruct"
n_items = 25
batch_size = 25

system_prompt = """
You will be presented with a sequence of words that follow a logical order, but one word in this sequence is missing, indicated by a question mark (?). Your task is to identify the missing word from a given set of options. To successfully complete this task, you should:

1. Analyze the sequence to understand the underlying logical pattern. Pay attention to the order of words, any repetitions, and how each word relates to the others in the sequence.
2. Do not rely on external tools or databases to analyze the sequence. Your reasoning should be based solely on the internal logic of the sequence as presented.
3. Consider the provided options carefully. There is only one correct answer that fits logically into the sequence in place of the question mark.
4. Present your answer in a clear and concise format: 'Answer: [chosen word]' (without the brackets). Include only your final choice in your response, without any additional explanation or text.

Your goal is to determine which option logically completes the sequence. Remember, the key to solving this puzzle is understanding the pattern that links the words in the sequence. Use this pattern to decide which of the options fits as the missing word.
"""

model = load_model(model_name)

/home/gopielka/gustaw/abs/cv_fv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 37/37 [00:21<00:00,  1.73it/s]


In [2]:
configs = {
    'nshot_0_ex_sysprompt': {'example_pattern': 'AABBAABB', 'system_prompt': system_prompt},
    'nshot_0_ex': {'example_pattern': 'AABBAABB', 'system_prompt': None},
    'nshot_0_sysprompt': {'example_pattern': None, 'system_prompt': system_prompt},
}
for n in range(4):
    configs['nshot_' + str(n)] = {'n_shot': n}


for name, config in configs.items():
    print('_'*10, name, '_'*10)
    dataset = AbstractTask(
        n=n_items, 
        batch_size=batch_size,
        tokenizer=model.tokenizer,
        **config
    )
    df = get_acc_df(model, dataset)

    path = RESULTS_DIR / model_name / f'{name}.csv'
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

__________ nshot_0_ex_sysprompt __________


100%|██████████| 48/48 [01:27<00:00,  1.83s/it]


__________ nshot_0_ex __________


100%|██████████| 48/48 [00:20<00:00,  2.31it/s]


__________ nshot_0_sysprompt __________


...

100%|██████████| 48/48 [00:21<00:00,  2.27it/s]


__________ nshot_2 __________


100%|██████████| 48/48 [00:28<00:00,  1.69it/s]


__________ nshot_3 __________


100%|██████████| 48/48 [00:35<00:00,  1.33it/s]
